# ORPHEUS Runtime

The notebook resolves the project from its current directory or ORPHEUS_PROJECT_DIR. GPU 0 is assigned to Qwen3-VL and GPU 1 to YuE.


In [1]:
import os
from pathlib import Path

# Set the path exactly to your cloned Kaggle directory
os.environ['ORPHEUS_PROJECT_DIR'] = '/kaggle/working/ORPHESUS'

def find_project_dir():
    configured = os.environ.get('ORPHEUS_PROJECT_DIR')
    candidates = [Path(configured).expanduser()] if configured else []
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents])
    
    for candidate in candidates:
        if (candidate / 'app' / 'server.py').is_file() and (candidate / 'web' / 'index.html').is_file():
            return candidate
            
    raise RuntimeError('ORPHEUS source files are not visible to this kernel. Open the notebook from the project folder or set ORPHEUS_PROJECT_DIR.')

PROJECT_DIR = find_project_dir()
os.chdir(PROJECT_DIR)
os.environ['ORPHEUS_QWEN_GPU'] = '0'
os.environ['ORPHEUS_YUE_GPU'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print(f'ORPHEUS project: {PROJECT_DIR}')


## Install application and YuE dependencies


In [2]:
!pip install -q -r requirements.txt
!sudo apt-get -qq update && sudo apt-get -qq install -y git-lfs
!git lfs install


In [3]:
import subprocess
import sys

yue_dir = PROJECT_DIR / 'YuE'
if not yue_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/multimodal-art-projection/YuE.git', str(yue_dir)], check=True)
codec_dir = yue_dir / 'inference' / 'xcodec_mini_infer'
if not codec_dir.exists():
    subprocess.run(['git', 'clone', 'https://huggingface.co/m-a-p/xcodec_mini_infer', str(codec_dir)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(yue_dir / 'requirements.txt')], check=True)
print('YuE inference:', yue_dir / 'inference' / 'infer.py')


## Verify the two T4 GPUs


In [4]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available in the connected kernel.')
if torch.cuda.device_count() < 2:
    raise RuntimeError('ORPHEUS requires two GPUs: Qwen on 0 and YuE on 1.')
for index in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(index)
    memory_gb = torch.cuda.get_device_properties(index).total_memory / 1024**3
    print(f'GPU {index}: {name} ({memory_gb:.1f} GB)')


## Load Qwen3-VL on GPU 0

This downloads `Qwen/Qwen3-VL-8B-Instruct` on first use and loads it in 4-bit mode.


In [ ]:
from app.qwen_engine import QwenEngine

qwen = QwenEngine()
allocated = torch.cuda.memory_allocated(0) / 1024**3
print(f'Qwen ready on GPU 0; allocated: {allocated:.2f} GB')
del qwen
torch.cuda.empty_cache()


## Preflight YuE on GPU 1


In [ ]:
from app.yue_engine import YuEEngine

yue = YuEEngine()
assert yue.ready, 'YuE inference script was not found.'
print(f'YuE ready: {yue.infer_script}')
print('YuE will use GPU 1 with two 30-second lyric sessions.')


## Start the Flask application


In [ ]:
import subprocess
import sys
import time
import requests

server_process = subprocess.Popen([sys.executable, '-B', '-m', 'app.server'], cwd=PROJECT_DIR)
for _ in range(15):
    try:
        health = requests.get('http://127.0.0.1:8000/api/health', timeout=2).json()
        break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError('Flask server did not start.')
print(health)


## End-to-end generation test

Run this final cell to generate a real short song. It downloads YuE checkpoints on first use and can take several minutes on T4 GPUs.


In [ ]:
from IPython.display import Audio, display

payload = {
    'text': 'Photosynthesis uses sunlight to convert carbon dioxide and water into glucose and oxygen.',
    'genre': 'uplifting pop',
    'mood': 'energetic',
    'language': 'English',
}
response = requests.post('http://127.0.0.1:8000/api/generate', json=payload, timeout=1800)
response.raise_for_status()
song = response.json()
assert song['success'] and song['audio_url']
audio_response = requests.get(f"http://127.0.0.1:8000{song['audio_url']}", timeout=60)
audio_response.raise_for_status()
assert audio_response.headers['Content-Type'].startswith('audio/')
print(song['title'])
display(Audio(data=audio_response.content, autoplay=False))
